In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import col, split
from pyspark.sql.functions import col
from pyspark.sql.functions import collect_set
from pyspark.sql.functions import count

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

sdf = spark.read.csv("hdfs://namenode:9000/data/raw/clima/", header=True, inferSchema=True)
sdf.show(5)

# https://www.geelongdataexchange.com.au/explore/dataset/rising-hf/information/

+-------------------+--------------------+------------+--------------------+-------------------+------------------+----------------------+---------------------+----------------+-----------+--------+
|      metadata_time|                name|      dev_id|            location|payload_fields_temp|payload_fields_hum|payload_fields_battery|payload_fields_period|skip_record_flag|temperature|humidity|
+-------------------+--------------------+------------+--------------------+-------------------+------------------+----------------------+---------------------+----------------+-----------+--------+
|2026-10-23 00:08:25|Spectacle Way, Le...|risinghf-59b|-38.19922446, 144...|               33.7|             47.71|                  3.41|              900 sec|               N|       33.7|   47.71|
|2026-10-23 00:08:01|Challis Avenue, B...|risinghf-5da|-38.29468833, 144...|              29.86|              64.8|                   3.5|              900 sec|               N|      29.86|    64.8|
|2026

In [2]:
	# Noticed that payload_fields_temp = temperature and payload_fields_hum = humidity for all rows
	# 	Decided to eliminate "temperature" and "humidity" columns
	# Noticed that payload_fields_period = "900 sec" for all rows
	# 	Decided to eliminate the "payload_fields_period" column
	# Noticed that skip_record_flag = 'N' for all rows
	# 	Decided to eliminate the "skip_record_flag" column
sdf = sdf.drop(
    "temperature",
    "humidity",
    "payload_fields_period",
    "skip_record_flag"
)

In [3]:
sdf.show(5)

+-------------------+--------------------+------------+--------------------+-------------------+------------------+----------------------+
|      metadata_time|                name|      dev_id|            location|payload_fields_temp|payload_fields_hum|payload_fields_battery|
+-------------------+--------------------+------------+--------------------+-------------------+------------------+----------------------+
|2026-10-23 00:08:25|Spectacle Way, Le...|risinghf-59b|-38.19922446, 144...|               33.7|             47.71|                  3.41|
|2026-10-23 00:08:01|Challis Avenue, B...|risinghf-5da|-38.29468833, 144...|              29.86|              64.8|                   3.5|
|2026-10-23 00:07:23|Mount Anakie, Anakie|risinghf-e47|-37.886444, 144.2...|              24.57|             61.87|                  3.01|
|2026-10-23 00:03:28|                NULL|risinghf-b02|                NULL|              33.74|             55.52|                  3.45|
|2026-10-23 00:00:09|Tower 

In [4]:
# Command Palette 
# print("sdf.show()")
# sdf.show()	# Displays the first 20 rows.
# print("sdf.show(5)")
# sdf.show(5) # Displays the first 5 rows.
print("sdf.printSchema()")
sdf.printSchema()	# Displays column names and data types.
print("sdf.columns")
sdf.columns	# Returns a list of column names.
print("sdf.dtypes")
sdf.dtypes	# Returns column names and data types.
print("sdf.count()")
sdf.count()	# Counts the number of rows.
print("print(len(sdf.columns))")
print(len(sdf.columns))	# Counts and prints the number of columns.
print("sdf.describe().show()")
sdf.describe().show()	# Shows basic statistics for numerical columns.
print("sdf.summary().show()")
sdf.summary().show()	# Shows more descriptive statistics.

sdf.printSchema()
root
 |-- metadata_time: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- dev_id: string (nullable = true)
 |-- location: string (nullable = true)
 |-- payload_fields_temp: double (nullable = true)
 |-- payload_fields_hum: double (nullable = true)
 |-- payload_fields_battery: double (nullable = true)

sdf.columns
sdf.dtypes
sdf.count()
print(len(sdf.columns))
7
sdf.describe().show()
+-------+--------------------+--------------------+--------------------+-------------------+------------------+----------------------+
|summary|                name|              dev_id|            location|payload_fields_temp|payload_fields_hum|payload_fields_battery|
+-------+--------------------+--------------------+--------------------+-------------------+------------------+----------------------+
|  count|              105145|              142064|              105145|             142064|            142064|                142064|
|   mean|                NULL|      

In [5]:
# Mostrar posibles pares (name, dev_id)
# name contiene nombres de lugares. dev_id contiene ids de sensores.
# Con este comando se sabe cuales lugares tienen más de un sensor.
print(sdf.select("name", "dev_id").distinct().count())
sdf.select("name", "dev_id").distinct().show(24)

print()
# Mostrar posibles pares (dev_id, name)
# Con este comando se sabe nota si un sensor se adjudica a varios lugares.
print(sdf.select("dev_id", "name").distinct().count())
sdf.select("dev_id", "name").distinct().show(24)



24
+--------------------+--------------------+
|                name|              dev_id|
+--------------------+--------------------+
|Seaspray Court, I...|        risinghf-260|
|Adella Court, Moolap|        risinghf-28f|
|Mount Anakie, Anakie|        risinghf-e47|
|Jeffrey Street, I...|        risinghf-96f|
|Spectacle Way, Le...|        risinghf-59b|
|Tower Road, Porta...|        risinghf-e54|
|Sophie Way, Armst...|        risinghf-d93|
|Challis Avenue, B...|        risinghf-5da|
|The Esplanade, Po...|        risinghf-13e|
|The Esplanade, In...|        risinghf-9cf|
|                NULL|          rising-e62|
|                NULL|        risinghf-e6b|
|                NULL|        risinghf-e47|
|                NULL|        risinghf-b02|
|                NULL|        risinghf-d58|
|                NULL|        risinghf-e54|
|                NULL|eui-8cf9574000002ddc|
|                NULL|        risinghf-59b|
|                NULL|        risinghf-d93|
|                NULL|       

In [6]:
# Dictionary like representation of the relationship between places and devices and vice versa
grouped_df = sdf.groupBy("name").agg(
    collect_set("dev_id").alias("dev_ids")
)

grouped_df.show(truncate=False)

print()

grouped_df_reversed = sdf.groupBy("dev_id").agg(
    collect_set("name").alias("names")
)

print(grouped_df_reversed.count())
grouped_df_reversed.show(truncate=False)
# grouped_df_reversed.show(18)

+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|name                         |dev_ids                                                                                                                                                                                                   |
+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Challis Avenue, Breamlea     |[risinghf-5da]                                                                                                                                                                                            |
|NULL                         |[risinghf-e47, risinghf-28f, 

In [7]:
grouped_df_location = sdf.groupBy("name").agg(
    collect_set("location").alias("locations")
)

grouped_df_location.show(truncate=False)


+-----------------------------+---------------------------+
|name                         |locations                  |
+-----------------------------+---------------------------+
|Challis Avenue, Breamlea     |[-38.29468833, 144.3927447]|
|NULL                         |[]                         |
|Adella Court, Moolap         |[-38.17258545, 144.4176005]|
|Spectacle Way, Leopold       |[-38.19922446, 144.459717] |
|Sophie Way, Armstrong Creek  |[-38.2391476, 144.3387083] |
|Jeffrey Street, Indented Head|[-38.13978826, 144.7099685]|
|Mount Anakie, Anakie         |[-37.886444, 144.269282]   |
|Tower Road, Portarlington    |[-38.123897, 144.641986]   |
|Seaspray Court, Indented Head|[-38.13990432, 144.7027301]|
|The Esplanade, Portarlington |[-38.11494912, 144.6725659]|
|The Esplanade, Indented Head |[-38.12428707, 144.6953381]|
+-----------------------------+---------------------------+



In [8]:
# Create PostreSQL tables
# Create Sensor table data frame.
sensors_df = sdf.select(
    "dev_id",
    "name",
    "location"
).distinct()
# Split location column
sensors_df = sensors_df.select(
    col("dev_id"),
    col("name"),
    split(col("location"), ",").getItem(0)
        .cast("double").alias("latitude"),
    split(col("location"), ",").getItem(1)
        .cast("double").alias("longitude")
)
# Eliminate null values from the sensor table. These are instead reconstructed for the measurements table.
sensors_df = sensors_df.filter(
    col("name").isNotNull() &
    col("latitude").isNotNull() &
    col("longitude").isNotNull()
)
# Exclude unnamed devices (determined above in this jupyter notebook)
excluded_devices = [
    "rising-e62",
    "risinghf-e6b",
    "risinghf-b02",
    "risinghf-d58",
    "eui-8cf9574000002ddc",
    "risinghf-6aa",
    "risinghf-4f1",
    "risinghf-e1c"
]
sensors_df = sensors_df.filter(
    ~col("dev_id").isin(excluded_devices)
)

sensors_df.printSchema()
sensors_df.show(24, truncate=False)

root
 |-- dev_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+------------+-----------------------------+------------+-----------+
|dev_id      |name                         |latitude    |longitude  |
+------------+-----------------------------+------------+-----------+
|risinghf-13e|The Esplanade, Portarlington |-38.11494912|144.6725659|
|risinghf-e47|Mount Anakie, Anakie         |-37.886444  |144.269282 |
|risinghf-d93|Sophie Way, Armstrong Creek  |-38.2391476 |144.3387083|
|risinghf-5da|Challis Avenue, Breamlea     |-38.29468833|144.3927447|
|risinghf-e54|Tower Road, Portarlington    |-38.123897  |144.641986 |
|risinghf-96f|Jeffrey Street, Indented Head|-38.13978826|144.7099685|
|risinghf-28f|Adella Court, Moolap         |-38.17258545|144.4176005|
|risinghf-59b|Spectacle Way, Leopold       |-38.19922446|144.459717 |
|risinghf-260|Seaspray Court, Indented Head|-38.13990432|144.7027301|

In [9]:
measurements_df = sdf.filter(
    ~col("dev_id").isin(excluded_devices)
).select(
    col("dev_id").cast("varchar(12)"),
    col("payload_fields_temp")
        .cast("double").alias("temperature"),
    col("payload_fields_hum")
        .cast("double").alias("humidity"),
    col("metadata_time")
        .cast("timestamp").alias("time"),
    col("payload_fields_battery")
        .cast("double").alias("battery")
)

measurements_df.printSchema()

print("Number of measurements:", measurements_df.count())
measurements_df.show(10, truncate=False)

# Check for nulls.
print("Row count with at least one null",
measurements_df.filter(
    col("dev_id").isNull() |
    col("temperature").isNull() |
    col("humidity").isNull() |
    col("time").isNull() |
    col("battery").isNull()
).count()
)

# Check for sensors having several simultaneous reads. i. e. primary key violations.
duplicate_measurements = measurements_df.groupBy(
    "dev_id", "time"
).agg(
    count("*").alias("records")
).filter(
    col("records") > 1
)
duplicate_measurements.show(10, truncate=False)

# Check that all dev_ids in the measurement table exist in the sensor table.
unknown_devices = measurements_df.join(
    sensors_df.select("dev_id"),
    on="dev_id",
    how="left_anti"
)
print(
    "Measurements with unknown devices:",
    unknown_devices.count()
)


root
 |-- dev_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- battery: double (nullable = true)

Number of measurements: 105496
+------------+-----------+--------+-------------------+-------+
|dev_id      |temperature|humidity|time               |battery|
+------------+-----------+--------+-------------------+-------+
|risinghf-59b|33.7       |47.71   |2026-10-23 00:08:25|3.41   |
|risinghf-5da|29.86      |64.8    |2026-10-23 00:08:01|3.5    |
|risinghf-e47|24.57      |61.87   |2026-10-23 00:07:23|3.01   |
|risinghf-e54|24.57      |66.27   |2026-10-23 00:00:09|3.13   |
|risinghf-5da|29.44      |66.27   |2026-10-22 23:52:45|3.5    |
|risinghf-e47|24.15      |63.34   |2026-10-22 23:52:16|3.01   |
|risinghf-e54|24.58      |67.24   |2026-10-22 23:45:03|3.12   |
|risinghf-59b|32.39      |50.64   |2026-10-22 23:37:57|3.41   |
|risinghf-5da|28.58      |68.22   |2026-10-22 23:37:28|3.49 

In [10]:
# Upload weather measurements to MongoDB.
MONGO_URI = "mongodb://mongo:27017"

(
    measurements_df.write
        .format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", "smartcities")
        .option("collection", "clima_measurements")
        .mode("overwrite")
        .save()
)

print()

In [11]:
mongo_df = (
    spark.read.format("mongodb")
        .option("connection.uri", MONGO_URI)
        .option("database", "smartcities")
        .option("collection", "clima_measurements")
        .load()
)

print("Documents in MongoDB:", mongo_df.count())

mongo_df.printSchema()

mongo_df.show(10, truncate=False)

print("measurements_df.count() = ", measurements_df.count())

Documents in MongoDB: 105496
root
 |-- _id: string (nullable = true)
 |-- battery: double (nullable = true)
 |-- dev_id: string (nullable = true)
 |-- humidity: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- time: timestamp (nullable = true)

+------------------------+-------+------------+--------+-----------+-------------------+
|_id                     |battery|dev_id      |humidity|temperature|time               |
+------------------------+-------+------------+--------+-----------+-------------------+
|6b0265f08939b727b6bf734b|3.18   |risinghf-59b|89.21   |11.08      |2026-04-19 15:03:39|
|6b0265f08939b727b6bf734e|3.01   |risinghf-e54|79.94   |14.7       |2026-04-19 14:57:43|
|6b0265f08939b727b6bf7350|3.35   |risinghf-28f|93.12   |9.3        |2026-04-19 14:55:14|
|6b0265f08939b727b6bf7352|3.2    |risinghf-5da|96.05   |9.26       |2026-04-19 14:49:35|
|6b0265f08939b727b6bf7354|2.89   |risinghf-e47|55.04   |15.98      |2026-04-19 14:49:17|
|6b0265f08939b727b6

In [12]:
jdbc_url = "jdbc:postgresql://postgres:5432/smartcities"

connection_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [13]:
sensors_df.write.jdbc(
    url=jdbc_url,
    table="sensors_table",
    mode="append",
    properties=connection_properties
)

In [14]:
# Verify by reading from PostgreSQL
# Can also verify with this
    # docker exec -it postgres psql -U postgres -d smartcities
    # SELECT * FROM sensors_table;

    # SELECT COUNT(*) FROM sensors_table;

sensors_from_db = spark.read.jdbc(
    url=jdbc_url,
    table="sensors_table",
    properties=connection_properties
)

sensors_from_db.show(truncate=False)

print("Rows in PostgreSQL:", sensors_from_db.count())

+------------+-----------------------------+------------+-----------+
|dev_id      |name                         |latitude    |longitude  |
+------------+-----------------------------+------------+-----------+
|risinghf-13e|The Esplanade, Portarlington |-38.11494912|144.6725659|
|risinghf-e47|Mount Anakie, Anakie         |-37.886444  |144.269282 |
|risinghf-d93|Sophie Way, Armstrong Creek  |-38.2391476 |144.3387083|
|risinghf-5da|Challis Avenue, Breamlea     |-38.29468833|144.3927447|
|risinghf-e54|Tower Road, Portarlington    |-38.123897  |144.641986 |
|risinghf-96f|Jeffrey Street, Indented Head|-38.13978826|144.7099685|
|risinghf-28f|Adella Court, Moolap         |-38.17258545|144.4176005|
|risinghf-59b|Spectacle Way, Leopold       |-38.19922446|144.459717 |
|risinghf-260|Seaspray Court, Indented Head|-38.13990432|144.7027301|
|risinghf-9cf|The Esplanade, Indented Head |-38.12428707|144.6953381|
+------------+-----------------------------+------------+-----------+

Rows in PostgreSQL: